In [ ]:
from langchain_groq import ChatGroq
from PyPDF2 import PdfReader
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
llm = ChatGroq(model='openai/gpt-oss-120b')

In [7]:
result = llm.invoke("Capital of India")

In [16]:
reader = PdfReader("sample_services_agreement.pdf")
text = ""
for page in reader.pages :
    text+=page.extract_text()

In [18]:
with open("extracted.txt", "w", encoding="utf-8") as f:
    f.write(text)

In [21]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size = 100 , chunk_overlap=20)
chunks = splitter.split_text(text)

In [26]:
import chromadb

In [28]:
client = chromadb.Client()
collection = client.get_or_create_collection('service_agreement')

In [29]:
collection.add(documents=chunks,
               ids=['c'+str(i) for i in range(len(chunks))])

In [32]:
question = 'what is late interest?'
result = collection.query(query_texts=[question],n_results=2)
context = result['documents'][0]
prompt = '''
I will provide you question and the context which may have answer of given question. 
You have to generate answer in 2 lines with respect to given context. If you dont find answer in given context, just reply "Answer Not found in given context".
question:{}
context:{}
'''.format(question,context)
result = llm.invoke(prompt)
print(result.content)

Late interest is the charge applied to overdue payments.  
It accrues at a rate of 3 % per month (or any part of a month).


137